# IT3170 region preprocessing v4 — Qwen2.5-VL-7B NF4
V4 uses bounded crop tasks, exact schema examples, board-by-board recognition, audited formatting repairs and at most two attempts per task.
Run canary pages 930/1042 before the remaining pilot. No unverified prediction is exported to CPT.


In [ ]:
import sys, subprocess, json, hashlib, zipfile, time, os, shutil
from pathlib import Path
from google.colab import drive,files
assert (3,10)<=sys.version_info[:2]<=(3,13), 'Use a compatible Python 3.10-3.13 runtime'
subprocess.run(['nvidia-smi'],check=True)
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/BKQuiz_region_v4');BASE.mkdir(parents=True,exist_ok=True)
uploaded=files.upload() # Choose region_v4_input.zip
if 'region_v4_input.zip' not in uploaded:raise ValueError('Select the v4 input ZIP')
data=uploaded.pop('region_v4_input.zip');archive=BASE/('input_'+hashlib.sha256(data).hexdigest()[:12]+'.zip')
if archive.exists():assert archive.read_bytes()==data
else:archive.write_bytes(data)
del data,uploaded
with zipfile.ZipFile(archive) as z:
    if z.testzip():raise ValueError('Corrupt ZIP')
    RUN_HASH=hashlib.sha256(z.read('manifest.json')).hexdigest()
    WORK=Path('/content')/('it3170_v4_'+RUN_HASH[:12]);WORK.mkdir(exist_ok=True)
    for member in z.infolist():
        if not (WORK/member.filename).resolve().is_relative_to(WORK.resolve()):raise ValueError('Unsafe ZIP member')
    z.extractall(WORK)
sys.path.insert(0,str(WORK))
from contracts import check_package
MANIFEST=check_package(WORK)
OUT=BASE/('results_'+RUN_HASH[:12]);OUT.mkdir(exist_ok=True)
os.environ['REGION_V4_CACHE']=str(BASE/'task_cache')
NEEDS_PADDLE=False # Specialist outputs are source-bound seeds; do not reinstall Paddle.
print('Manifest:',RUN_HASH,'Parents:',len(MANIFEST['requests']),'Tasks:',len(MANIFEST['tasks']))
print('Persistent output:',OUT)


## Restore only the VLM environment
Reuses Python-matched archives in MyDrive/BKQuiz_preprocessing when compatible; checks locked dependencies afterwards.
No Paddle environment is required. Weights are separate from the environment archive.


In [ ]:
PADDLE_ENV=Path('/content/it3170_paddle_env');VLM_ENV=Path('/content/it3170_vlm_env')
ENV_CACHE=Path('/content/drive/MyDrive/BKQuiz_preprocessing')
tag=f'py{sys.version_info.major}{sys.version_info.minor}'
setup_start=time.perf_counter()
def runnable(env):
    py=env/'bin/python'
    if not py.exists():return False
    return subprocess.run([str(py),'-c','import pip'],capture_output=True).returncode==0
for env in ([PADDLE_ENV] if NEEDS_PADDLE else [])+[VLM_ENV]:
    if runnable(env):continue
    if env.exists() or env.is_symlink():env.rename(env.with_name(env.name+'_broken_'+str(time.time_ns())))
    archive=ENV_CACHE/f'{env.name}_{tag}_v1.tar.gz'
    if archive.exists():
        local=Path('/content')/archive.name;shutil.copy2(archive,local)
        listing=subprocess.check_output(['tar','-tzf',str(local)],text=True).splitlines()
        if any(Path(n).is_absolute() or '..' in Path(n).parts or Path(n).parts[0]!=env.name for n in listing):raise ValueError('Unsafe venv archive')
        subprocess.run(['tar','-xzf',str(local),'-C','/content'],check=True)
    if not runnable(env):
        if env.exists():env.rename(env.with_name(env.name+'_incompatible_'+str(time.time_ns())))
        subprocess.run([sys.executable,'-m','venv','--without-pip',str(env)],check=True)
        subprocess.run([sys.executable,'-m','pip','--python',str(env/'bin/python'),'install','pip==25.2'],check=True)
PADDLE_PY=str(PADDLE_ENV/'bin/python');VLM_PY=str(VLM_ENV/'bin/python')
def versions_match(py,expected):
    source="import importlib.metadata as m,json;print(json.dumps({k:m.version(k).split('+')[0] for k in "+repr(list(expected))+"}))"
    p=subprocess.run([py,'-c',source],capture_output=True,text=True)
    return p.returncode==0 and json.loads(p.stdout)==expected
paddle_versions={'paddleocr':'3.7.0','paddlex':'3.7.2','paddlepaddle-gpu':'3.2.2'}
if NEEDS_PADDLE and not versions_match(PADDLE_PY,paddle_versions):
    subprocess.run([PADDLE_PY,'-m','pip','install','paddlepaddle-gpu==3.2.2','--index-url','https://www.paddlepaddle.org.cn/packages/stable/cu118/'],check=True)
    subprocess.run([PADDLE_PY,'-m','pip','install','paddleocr[doc-parser]==3.7.0','paddlex[ocr]==3.7.2','Pillow==11.3.0'],check=True)
vlm_versions={'torch':'2.6.0','torchvision':'0.21.0','transformers':'4.57.6','accelerate':'1.12.0','qwen-vl-utils':'0.0.14'}
if not versions_match(VLM_PY,vlm_versions):
    subprocess.run([VLM_PY,'-m','pip','install','torch==2.6.0','torchvision==0.21.0','--index-url','https://download.pytorch.org/whl/cu124/'],check=True)
    subprocess.run([VLM_PY,'-m','pip','install','transformers==4.57.6','accelerate==1.12.0','qwen-vl-utils==0.0.14','Pillow==11.3.0'],check=True)
if not versions_match(VLM_PY,{'bitsandbytes':'0.48.1'}):
    subprocess.run([VLM_PY,'-m','pip','install','bitsandbytes==0.48.1'],check=True)
if NEEDS_PADDLE:subprocess.run([PADDLE_PY,'-c','import paddle; paddle.utils.run_check(); assert paddle.is_compiled_with_cuda()'],check=True)
subprocess.run([VLM_PY,'-c','import torch,bitsandbytes; assert torch.cuda.is_available(); print(torch.ones(1,device="cuda"))'],check=True)
(OUT/('setup_seconds_'+str(time.time_ns())+'.json')).write_text(json.dumps({'seconds':time.perf_counter()-setup_start}))
for name,py in ([('paddle',PADDLE_PY)] if NEEDS_PADDLE else [])+[('vlm',VLM_PY)]:
    (OUT/(name+'_freeze_'+str(time.time_ns())+'.txt')).write_text(subprocess.check_output([py,'-m','pip','freeze'],text=True))


In [ ]:
def run(command,extra=None):
    env=dict(os.environ)
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
    subprocess.run([VLM_PY,str(WORK/'worker.py'),command,'--input',str(WORK),'--out',str(OUT)]+(extra or []),env=env,check=True)
run('collect')


## Memory canary: pages 930 and 1042
Actual CUDA telemetry is logged per attempt, including failures. First profile: 384 visual tokens, bounded text; retry: 256.
This does not guarantee success or semantic accuracy on T4. Read the collected result before proceeding.


In [ ]:
run('vlm',['--scope','canary','--retry-failed'])
def latest():
    candidates=[json.loads(p.read_text()) for p in OUT.glob('results_*.json')]
    return max(candidates,key=lambda x:(sum(t['status']=='ok' for t in x['tasks']),len(x['tasks']),sum(r['status']!='not_run' for r in x['records'])))
CANARY=latest()
for r in CANARY['records']:
    if r['engine']=='vlm' and r['source']['page'] in MANIFEST['canary_pages']:
        print(r['id'],r['status'],r.get('quality',{}).get('flags'))
CANARY_OK=all(r['status']=='ok' for r in CANARY['records'] if r['engine']=='vlm' and r['source']['page'] in MANIFEST['canary_pages'])
print('Canary schema success:',CANARY_OK)


## Remaining pilot: 129, 494 and 812
Successful canary tasks are resumed from disk/cache. Do not bypass the canary failure gate by running the full batch.


In [ ]:
if CANARY_OK:
    run('vlm',['--scope','pilot','--retry-failed'])
else:
    print('Stop expansion: download diagnostics below and inspect canary errors.')
PILOT=latest()
for r in PILOT['records']:
    if r['engine']=='vlm' and r['source']['page'] in MANIFEST['pilot_pages']:
        print(r['id'],r['status'],r.get('quality',{}).get('flags'))


## Optional full run
Set RUN_FULL=True only after source review of the pilot. Structural success does not verify graph facts or board states.
Retries are bounded per invocation; failed attempts are preserved. Do not repeatedly retry without inspecting their cause.


In [ ]:
RUN_FULL=False
if RUN_FULL:run('vlm',['--scope','all','--retry-failed'])
else:print('Pilot only. Unexecuted tasks remain not_run.')


## Download results, including failures
Run this cell even if inference failed. All snapshots, task attempts, raw model output, repairs and telemetry are kept.


In [ ]:
run('collect')
result_zip=BASE/('region_v4_results_'+RUN_HASH[:12]+'_'+str(time.time_ns())+'.zip')
with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():z.write(p,p.relative_to(OUT).as_posix())
print(result_zip)
files.download(str(result_zip))
